# Qwen2.5-VL Fashion Caption Generation (Quality Pilot)

Caption-source ablation: generate fashion-focused captions with `Qwen/Qwen2.5-VL-7B-Instruct` as a candidate replacement for LLaVA captions.

**Do not run full-dataset generation until a prompt passes manual quality review.**

Completed workflow in this notebook:
1. build a small **balanced evaluation sample** (optionally including hard/confusable styles)
2. generate captions with **3–4 prompt variants** (B, C, D recommended; A optional baseline)
3. save a **comparison CSV** and inspect captions side by side with LLaVA
4. score captions manually with a 1–5 rubric
5. lock `SELECTED_PROMPT_ID`
6. run a **second validation sample** (5 images per class) on the chosen prompt only
7. only then generate captions for the full dataset

Outputs: `data/captions/`

**Environment (important):** Use the dedicated conda env **`qwen_vl`**, not `tf_gpu`.

- `tf_gpu` has PyTorch 2.1.2 + transformers 4.37.2 for your fusion/CLIP training.
- Qwen2.5-VL needs **PyTorch >= 2.4** and **transformers >= 4.49**.
- Installing Qwen packages into `tf_gpu` will break training notebooks.

In Jupyter: **Kernel → Change Kernel → Python (qwen_vl)**

One-time setup (terminal):

```bash
conda create -n qwen_vl python=3.10 -y
conda activate qwen_vl
pip install torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
pip install "transformers>=4.49,<4.54" accelerate qwen-vl-utils pandas pillow tqdm ipykernel
python -m ipykernel install --user --name qwen_vl --display-name "Python (qwen_vl)"
```

**Important:** Prompts must describe visual/style cues only. Do **not** ask the model to predict one of the 14 dataset labels directly.

In [1]:
import sys
from packaging import version

REQUIRED_ENV = "qwen_vl"
MIN_TORCH = "2.4.0"
MIN_TRANSFORMERS = "4.49.0"

print("Python:", sys.executable)

if REQUIRED_ENV not in sys.executable:
    raise RuntimeError(
        f"Wrong kernel. This notebook must run in conda env '{REQUIRED_ENV}'.\n"
        f"Current: {sys.executable}\n"
        "In Jupyter: Kernel → Change Kernel → Python (qwen_vl)\n"
        "Do NOT pip install Qwen/transformers upgrades into tf_gpu."
    )

import torch
import transformers
from transformers.utils import is_torch_available

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("is_torch_available:", is_torch_available())

if version.parse(torch.__version__.split("+")[0]) < version.parse(MIN_TORCH):
    raise RuntimeError(f"Need torch>={MIN_TORCH}, found {torch.__version__}")
if version.parse(transformers.__version__) < version.parse(MIN_TRANSFORMERS):
    raise RuntimeError(f"Need transformers>={MIN_TRANSFORMERS}, found {transformers.__version__}")
if not is_torch_available():
    raise RuntimeError(
        "transformers cannot see PyTorch (is_torch_available=False). "
        "Usually torch is too old or the wrong env is active."
    )

from transformers import Qwen2_5_VLForConditionalGeneration  # noqa: F401
print("Environment OK for Qwen2.5-VL.")

Python: /home/ding-zhang/anaconda3/envs/qwen_vl/bin/python


/home/ding-zhang/anaconda3/envs/qwen_vl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.4.1+cu121 | cuda: True
transformers: 4.53.3
is_torch_available: True
Environment OK for Qwen2.5-VL.


## 1. Configuration

In [2]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote
import json
import random
import time
import warnings

import pandas as pd
import torch
from IPython.display import display, HTML
from tqdm import tqdm

warnings.filterwarnings("ignore")

_walk = Path.cwd().resolve()
for _ in range(10):
    if (_walk / "experiments").is_dir() and (_walk / "data").is_dir():
        PROJECT_ROOT = _walk
        break
    _walk = _walk.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

BASE_DIR = PROJECT_ROOT
DATASET_CSV = PROJECT_ROOT / "data" / "processed" / "caption_dataset_final_full.csv"
CAPTIONS_DIR = PROJECT_ROOT / "data" / "captions"
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
CAPTION_MODEL_NAME = "qwen2.5-vl-7b-instruct"
GENERATION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

# --- pilot stage controls ---
# "smoke"     -> 3 images/class, compare prompts B/C/D (42 images)
# "validation"-> 5 images/class, selected prompt only (70 images)
# "full"      -> all images, selected prompt only
PILOT_STAGE = "smoke"

SAMPLE_SEED = 42
IMAGES_PER_STYLE_SMOKE = 3       # 14 * 3 = 42
IMAGES_PER_STYLE_VALIDATION = 5  # 14 * 5 = 70

# Which prompts to run in smoke stage (keep small: 3-4 max)
# SMOKE_PROMPT_IDS = ["prompt_b", "prompt_c", "prompt_d"]
# Optional baseline; add "prompt_a" if you want a neutral comparator
SMOKE_PROMPT_IDS = ["prompt_a", "prompt_b", "prompt_c", "prompt_d"]

# Set here before validation/full runs (re-running from top resets this cell).
SELECTED_PROMPT_ID = None  # e.g. "prompt_c" after smoke manual review

MAX_NEW_TOKENS = 180
TEMPERATURE = 0.2
TOP_P = 0.9
TARGET_WORDS_MIN = 40
TARGET_WORDS_MAX = 80

RUN_GENERATION = True  # set False to skip GPU generation cells

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PILOT_STAGE:", PILOT_STAGE)
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_ROOT: /home/ding-zhang/Dongmei/DATA255/Project
PILOT_STAGE: smoke
DEVICE: cuda
GPU: NVIDIA GeForce RTX 4090


## 2. Prompt Candidates (No Direct Label Prediction)

All prompts ask for **visual/style cues**, not "which of the 14 classes is this?"

Avoid prompts like: *"What fashion style is this?"* or *"Is this girlish, fairy, lolita..."*

Good captions: garment types, silhouette, colors, texture, layering, accessories, aesthetic mood.
Risky captions: *"this is a lolita outfit"* (explicit dataset-label leakage).

In [3]:
PROMPT_CANDIDATES = {
    "prompt_a": (
        "Describe this fashion image in detail. Focus on the clothing items, colors, patterns, "
        "materials, silhouette, accessories, and overall aesthetic. Avoid describing the background "
        "unless it is relevant to the outfit. Write one concise paragraph of 40-80 words."
    ),
    "prompt_b": (
        "You are analyzing a fashion outfit for style classification. Describe only the visible "
        "fashion-relevant details: clothing items, silhouette and fit, color palette, patterns, "
        "fabric or texture, layering, accessories, and overall aesthetic mood. Avoid describing "
        "the background, pose, face, camera angle, or image quality unless it affects the outfit. "
        "Do not directly guess the dataset label. Write one concise paragraph of 40-80 words."
    ),
    "prompt_c": (
        "Analyze the outfit for fashion style recognition. Describe: 1) clothing items, "
        "2) silhouette and fit, 3) colors and patterns, 4) fabrics or textures, 5) accessories and "
        "styling details, and 6) overall fashion mood. Do not guess the dataset label directly. "
        "Keep the description factual and image-grounded. Limit the answer to 40-80 words."
    ),
    "prompt_d": (
        "Describe the outfit in a way that helps distinguish its fashion style from similar styles. "
        "Focus on visible clothing, color palette, silhouette, texture, accessories, and aesthetic cues. "
        "Avoid generic scene description. Do not name the dataset class label. "
        "Write 2-3 concise sentences (about 40-80 words total)."
    ),
}

DATASET_STYLE_LABELS = [
    "conservative", "dressy", "ethnic", "fairy", "feminine", "gal", "girlish",
    "kireime-casual", "retro", "natural", "rock", "street", "lolita", "mode",
]

# Pairs often confused — sample extra images from these styles in the pilot set
CONFUSABLE_STYLE_GROUPS = [
    ["fairy", "lolita"],
    ["girlish", "feminine"],
    ["kireime-casual", "natural"],
    ["street", "rock"],
]

for pid in SMOKE_PROMPT_IDS:
    print(f"[{pid}]\n{PROMPT_CANDIDATES[pid]}\n")

[prompt_a]
Describe this fashion image in detail. Focus on the clothing items, colors, patterns, materials, silhouette, accessories, and overall aesthetic. Avoid describing the background unless it is relevant to the outfit. Write one concise paragraph of 40-80 words.

[prompt_b]
You are analyzing a fashion outfit for style classification. Describe only the visible fashion-relevant details: clothing items, silhouette and fit, color palette, patterns, fabric or texture, layering, accessories, and overall aesthetic mood. Avoid describing the background, pose, face, camera angle, or image quality unless it affects the outfit. Do not directly guess the dataset label. Write one concise paragraph of 40-80 words.

[prompt_c]
Analyze the outfit for fashion style recognition. Describe: 1) clothing items, 2) silhouette and fit, 3) colors and patterns, 4) fabrics or textures, 5) accessories and styling details, and 6) overall fashion mood. Do not guess the dataset label directly. Keep the descrip

## 3. Manual Quality Rubric (Use Before Choosing A Prompt)

| Score | Meaning |
|-------|--------|
| 1 | Mostly wrong or irrelevant |
| 2 | Generic clothing description, little style value |
| 3 | Correct basic outfit description |
| 4 | Good fashion/style cues |
| 5 | Highly discriminative and useful for style classification |

Check each caption on:
- visual correctness
- fashion detail
- style-discriminative value
- avoidance of background noise
- consistency across similar images
- hallucination level

The caption does **not** need to name the true class. It should provide cues that help separate similar styles.

Choose the prompt with the best **usefulness**, not the longest caption.

## 4. Load Dataset, Resolve Paths, Build Evaluation Sample

In [4]:
def resolve_image_path(image_path, base_dir):
    image_path = unquote(str(image_path))
    p = Path(image_path)
    if p.is_file():
        return str(p)
    rel_path = image_path.replace("\\", "/")
    if rel_path.startswith("dataset/"):
        candidate = base_dir / "data" / "raw dataset" / rel_path[len("dataset/"):]
        if candidate.is_file():
            return str(candidate)
    candidate = base_dir / rel_path
    if candidate.is_file():
        return str(candidate)
    return str(p)


df_all = pd.read_csv(DATASET_CSV)
if "status" in df_all.columns:
    df_all = df_all[df_all["status"] == "success"].copy()

df_all["resolved_path"] = df_all["image_path"].map(lambda p: resolve_image_path(p, BASE_DIR))
df_all["path_exists"] = df_all["resolved_path"].map(lambda p: Path(p).is_file())
df_work = df_all[df_all["path_exists"]].copy().reset_index(drop=True)
print(f"Usable rows: {len(df_work)} | styles: {df_work['style'].nunique()}")

Usable rows: 13196 | styles: 14


In [5]:
def build_balanced_sample(df, images_per_style, seed=42, boost_confusable=True):
    """Stratified sample: N images per style. Optionally add 1 extra per confusable style."""
    rng = random.Random(seed)
    confusable_styles = {s for group in CONFUSABLE_STYLE_GROUPS for s in group}
    selected_idx = []

    for style, group in df.groupby("style", sort=True):
        idxs = list(group.index)
        rng.shuffle(idxs)
        n_take = images_per_style + (1 if boost_confusable and style in confusable_styles else 0)
        n_take = min(n_take, len(idxs))
        selected_idx.extend(idxs[:n_take])

    out = df.loc[sorted(set(selected_idx))].copy().reset_index(drop=True)
    out["true_style"] = out["style"]
    out["llava_caption"] = out["caption"]
    return out


if PILOT_STAGE == "smoke":
    df_eval = build_balanced_sample(
        df_work, IMAGES_PER_STYLE_SMOKE, seed=SAMPLE_SEED, boost_confusable=True
    )
    eval_tag = f"smoke_{IMAGES_PER_STYLE_SMOKE}per_style"
elif PILOT_STAGE == "validation":
    if SELECTED_PROMPT_ID is None:
        raise RuntimeError("Set SELECTED_PROMPT_ID before validation stage.")
    df_eval = build_balanced_sample(
        df_work, IMAGES_PER_STYLE_VALIDATION, seed=SAMPLE_SEED + 7, boost_confusable=True
    )
    eval_tag = f"validation_{IMAGES_PER_STYLE_VALIDATION}per_style"
elif PILOT_STAGE == "full":
    df_eval = None
    eval_tag = "full"
    print("PILOT_STAGE='full': no pilot sample here. Full captions use Section 8 (checkpoint/resume).")
else:
    raise ValueError(f"Unknown PILOT_STAGE: {PILOT_STAGE}")

if df_eval is not None:
    print(f"Evaluation sample [{eval_tag}]: {len(df_eval)} images")
    print(df_eval["style"].value_counts().sort_index())
    print(
        "Confusable styles in sample:",
        sorted(set(df_eval["style"]) & {s for g in CONFUSABLE_STYLE_GROUPS for s in g}),
    )

Evaluation sample [smoke_3per_style]: 50 images
style
conservative      3
dressy            3
ethnic            3
fairy             4
feminine          4
gal               3
girlish           4
kireime-casual    4
lolita            4
mode              3
natural           4
retro             3
rock              4
street            4
Name: count, dtype: int64
Confusable styles in sample: ['fairy', 'feminine', 'girlish', 'kireime-casual', 'lolita', 'natural', 'rock', 'street']


## 5. Load Qwen2.5-VL And Generate Captions

Smoke and validation captions run here. **Full-dataset generation is only in Section 8** (`run_full_with_resume`).

In [6]:
model = None
processor = None

if RUN_GENERATION:
    try:
        from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
        from qwen_vl_utils import process_vision_info
    except Exception as exc:
        raise ImportError(
            "Qwen2.5-VL import failed. Use conda env qwen_vl (torch>=2.4, transformers>=4.49). "
            "Do not upgrade packages inside tf_gpu."
        ) from exc

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=dtype, device_map="auto" if torch.cuda.is_available() else None
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model.eval()
    print("Model loaded:", MODEL_ID)

Loading checkpoint shards: 100%|██████████| 5/5 [00:01<00:00,  4.18it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Model loaded: Qwen/Qwen2.5-VL-7B-Instruct


In [7]:
def generate_qwen_caption(image_path, prompt_text, max_new_tokens=MAX_NEW_TOKENS):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{Path(image_path).resolve()}"},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt"
    ).to(model.device)

    t0 = time.time()
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=TEMPERATURE > 0,
            temperature=TEMPERATURE if TEMPERATURE > 0 else None,
            top_p=TOP_P if TEMPERATURE > 0 else None,
        )
    elapsed = time.time() - t0
    trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    caption = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    return caption, elapsed


def detect_label_leak(caption):
    """Substring match on class names; may flag descriptive words (e.g. 'natural palette').
    Use as a warning only, not an automatic rejection rule."""
    text = caption.lower()
    hits = [lbl for lbl in DATASET_STYLE_LABELS if lbl in text]
    return hits


def caption_quality_helpers(caption):
    words = caption.split()
    wc = len(words)
    label_hits = detect_label_leak(caption)
    return {
        "word_count": wc,
        "in_target_word_range": TARGET_WORDS_MIN <= wc <= TARGET_WORDS_MAX,
        "label_leak_hits": ",".join(label_hits) if label_hits else "",
        "has_label_leak": bool(label_hits),
    }

In [8]:
def run_caption_batch(df_sample, prompt_id, prompt_text):
    rows = []
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample), desc=prompt_id):
        try:
            caption, elapsed = generate_qwen_caption(row["resolved_path"], prompt_text)
            status = "success"
            error = ""
        except Exception as exc:
            caption, elapsed, status, error = "", None, "error", str(exc)

        meta = caption_quality_helpers(caption) if caption else {
            "word_count": 0, "in_target_word_range": False, "label_leak_hits": "", "has_label_leak": False
        }
        rows.append({
            "image_path": row["image_path"],
            "true_style": row["true_style"],
            "llava_caption": row.get("llava_caption", ""),
            "caption_model": CAPTION_MODEL_NAME,
            "prompt_id": prompt_id,
            "prompt_text": prompt_text,
            "prompt_version": prompt_id,
            "generation_date": GENERATION_DATE,
            "caption": caption,
            "status": status,
            "error": error,
            "generation_time_sec": elapsed,
            **meta,
        })
    return pd.DataFrame(rows)


if not RUN_GENERATION:
    print("RUN_GENERATION=False; skipping.")
elif model is None:
    raise RuntimeError("Load model in Section 5 first.")
elif PILOT_STAGE == "full":
    print(
        "PILOT_STAGE='full': skipping pilot batch generation. "
        "Use Section 8 run_full_with_resume() for checkpointed full-dataset captions."
    )
else:
    if PILOT_STAGE == "smoke":
        prompt_ids = SMOKE_PROMPT_IDS
    elif PILOT_STAGE == "validation":
        if SELECTED_PROMPT_ID is None:
            raise RuntimeError("Set SELECTED_PROMPT_ID for validation stage.")
        prompt_ids = [SELECTED_PROMPT_ID]
    else:
        raise ValueError(PILOT_STAGE)

    if prompt_ids:
        frames = [
            run_caption_batch(df_eval, pid, PROMPT_CANDIDATES[pid]) for pid in prompt_ids
        ]
        df_pilot = pd.concat(frames, ignore_index=True)

        pilot_path = CAPTIONS_DIR / f"qwen25vl_{eval_tag}_comparison.csv"
        df_pilot.to_csv(pilot_path, index=False)
        print("Saved:", pilot_path)

        if PILOT_STAGE == "smoke":
            summary = (
                df_pilot[df_pilot["status"] == "success"]
                .groupby("prompt_id", as_index=False)
                .agg(
                    n=("caption", "count"),
                    mean_word_count=("word_count", "mean"),
                    pct_in_word_range=("in_target_word_range", "mean"),
                    pct_label_leak=("has_label_leak", "mean"),
                    mean_gen_time_sec=("generation_time_sec", "mean"),
                )
                .sort_values(["pct_label_leak", "pct_in_word_range"], ascending=[True, False])
            )
            display(summary)

prompt_d: 100%|██████████| 50/50 [01:21<00:00,  1.64s/it]

Saved: /home/ding-zhang/Dongmei/DATA255/Project/data/captions/qwen25vl_smoke_3per_style_comparison.csv


,prompt_id,n,mean_word_count,pct_in_word_range,pct_label_leak,mean_gen_time_sec
2,prompt_c,50,114.78,0.06,0.26,2.936826
1,prompt_b,50,83.14,0.40,0.42,2.025900
0,prompt_a,50,65.48,0.92,0.54,1.616643
3,prompt_d,50,66.52,0.90,0.60,1.633145


## 6. Side-By-Side Review (Start With Hard Examples)

Inspect confusable-style pairs first: fairy/lolita, girlish/feminine, kireime-casual/natural, street/rock.

In [9]:
def show_side_by_side(df_compare, image_path, max_chars=800):
    subset = df_compare[df_compare["image_path"] == image_path]
    if subset.empty:
        return
    style = subset.iloc[0]["true_style"]
    llava = subset.iloc[0].get("llava_caption", "")
    html = [f"<h3>{style} | {Path(image_path).name}</h3>", f"<p><b>LLaVA</b><br>{llava[:max_chars]}</p>"]
    for _, r in subset.sort_values("prompt_id").iterrows():
        cap = r["caption"] if r["status"] == "success" else f"[ERROR] {r['error']}"
        leak = r.get("label_leak_hits", "")
        leak_note = f" | label_leak={leak}" if leak else ""
        html.append(
            f"<p><b>{r['prompt_id']}</b> ({r.get('word_count', '?')} words{leak_note})<br>{cap[:max_chars]}</p>"
        )
    display(HTML("".join(html)))


if "df_pilot" in globals():
    confusable_styles = {s for g in CONFUSABLE_STYLE_GROUPS for s in g}
    hard_images = (
        df_eval[df_eval["style"].isin(confusable_styles)]["image_path"].drop_duplicates().tolist()
    )
    print(f"Reviewing {min(12, len(hard_images))} hard/confusable images first...")
    for img in hard_images[:12]:
        show_side_by_side(df_pilot, img)
else:
    print("Run Section 5 first.")

Reviewing 12 hard/confusable images first...


## 7. Manual Scoring Template

Export a blank scoring sheet, fill `manual_score_1to5` offline or in Excel, then pick the best `prompt_id`.

In [ ]:
if "df_pilot" in globals() and PILOT_STAGE == "smoke":
    scoring = df_pilot[["image_path", "true_style", "prompt_id", "caption", "word_count", "has_label_leak"]].copy()
    scoring["manual_score_1to5"] = ""
    scoring["notes"] = ""
    score_path = CAPTIONS_DIR / f"qwen25vl_{eval_tag}_manual_scoring_template.csv"
    scoring.to_csv(score_path, index=False)
    print("Manual scoring template:", score_path)
    print("After scoring, set SELECTED_PROMPT_ID and PILOT_STAGE='validation'.")

## 8. Lock Prompt → Validation → Full Dataset

1. Set `SELECTED_PROMPT_ID` in **Section 1** (e.g. `"prompt_c"`) — rerunning from the top resets the config cell.
2. Set `PILOT_STAGE = "validation"` and rerun Sections 4–5 (70 images, one prompt).
3. Set `PILOT_STAGE = "full"`, rerun Sections 1–5 (model load only; generation cell skips), then Section 8 for checkpointed full run (~13k images).

In [ ]:
# --- edit after manual review ---
# SELECTED_PROMPT_ID = "prompt_c"
# PILOT_STAGE = "validation"   # then "full"

CHECKPOINT_EVERY = 200


def run_full_with_resume(df_images, prompt_id, prompt_text, output_csv):
    done = set()
    if output_csv.exists():
        prev = pd.read_csv(output_csv)
        done = set(prev.loc[prev["status"] == "success", "image_path"].astype(str))
        print(f"Resume: {len(done)} captions already saved.")

    rows = []
    for _, row in tqdm(df_images.iterrows(), total=len(df_images), desc="full"):
        if str(row["image_path"]) in done:
            continue
        try:
            caption, elapsed = generate_qwen_caption(row["resolved_path"], prompt_text)
            status, error = "success", ""
        except Exception as exc:
            caption, elapsed, status, error = "", None, "error", str(exc)
        meta = caption_quality_helpers(caption) if caption else {
            "word_count": 0,
            "in_target_word_range": False,
            "label_leak_hits": "",
            "has_label_leak": False,
        }
        rows.append({
            "image_path": row["image_path"],
            "style": row["style"],
            "true_style": row["style"],
            "caption_model": CAPTION_MODEL_NAME,
            "prompt_id": prompt_id,
            "prompt_text": prompt_text,
            "prompt_version": prompt_id,
            "generation_date": GENERATION_DATE,
            "caption": caption,
            "status": status,
            "error": error,
            "generation_time_sec": elapsed,
            **meta,
        })
        if len(rows) >= CHECKPOINT_EVERY:
            pd.DataFrame(rows).to_csv(output_csv, mode="a", header=not output_csv.exists(), index=False)
            rows = []
    if rows:
        pd.DataFrame(rows).to_csv(output_csv, mode="a", header=not output_csv.exists(), index=False)
    print("Saved:", output_csv)


def audit_full_caption_csv(full_csv, dedupe_if_needed=True):
    """Post-run checks: success rate, duplicates; keep latest successful row per image."""
    if not full_csv.exists():
        print("File not found:", full_csv)
        return None
    full_df = pd.read_csv(full_csv)
    print("Rows:", len(full_df))
    print("Unique images:", full_df["image_path"].nunique())
    print("Success rate:", (full_df["status"] == "success").mean())
    dup_count = int(full_df.duplicated("image_path", keep=False).sum())
    print("Duplicate rows:", dup_count)
    if "has_label_leak" in full_df.columns:
        ok = full_df["status"] == "success"
        print("Label-leak warning rate (success only):", full_df.loc[ok, "has_label_leak"].mean())
    if "word_count" in full_df.columns:
        ok = full_df["status"] == "success"
        print("Mean word count (success only):", full_df.loc[ok, "word_count"].mean())
    if dup_count and dedupe_if_needed:
        success_df = full_df[full_df["status"] == "success"].copy()
        other_df = full_df[full_df["status"] != "success"].copy()
        deduped_success = success_df.drop_duplicates("image_path", keep="last")
        full_df = pd.concat([deduped_success, other_df], ignore_index=True)
        full_df = full_df.drop_duplicates("image_path", keep="last")
        full_df.to_csv(full_csv, index=False)
        print("Deduplicated file saved:", full_csv)
        print("Rows after dedupe:", len(full_df))
    return full_df


if PILOT_STAGE == "full" and RUN_GENERATION and SELECTED_PROMPT_ID:
    full_csv = CAPTIONS_DIR / f"qwen25vl_{SELECTED_PROMPT_ID}_captions.csv"
    run_full_with_resume(df_work, SELECTED_PROMPT_ID, PROMPT_CANDIDATES[SELECTED_PROMPT_ID], full_csv)
    audit_full_caption_csv(full_csv, dedupe_if_needed=True)
elif PILOT_STAGE == "full":
    print("Set SELECTED_PROMPT_ID and RUN_GENERATION=True for full dataset run.")